<h1 align = "center">
Rwanda Maize-Urea Trial
</h1>

<h2 align = "center">
Dataset Preparation & Cleaning
</h2> 

# Objectives

By the end of the preparation and cleaning process, we want to have prepared datasets that will be suitable for analysis without silently changing the meaning of original observations. We want to

- **preserve source data**; keep the original field, plot and soil DataFramesunchanged, and only create working copies for preparation.

- **Standardize data types**; Parse data into proper datetime columns, verify numeric fields and standardize categorical values only when justified.

- **Establish safe relationships**; Resolve or flag unmatched field-plot records, retain incomplete experimental units, and verify keys before merging.

- **Preserve uncertainty**; Retain missing values and source fertilizer totals when their meaning or cause is unknown. Create explicit flags where useful.

- **Validate the prepared data**; Check row counts, keys, relationships, treatment representation, date logic, and the effects of any transformations.


## Working Copies and Baseline Checks

What we don't want is to find ourselves making the classical mistake of cleaning the dataset in place and losing track of which values came from the original source.

- For a research project, we should be able to explain exactly what changed and be able to reproduce our preparation process

We'll, therefore, create separate DataFrames and record their initial dimensions, column names, data types, and missing value-counts

Let's create working copies and record baseline dataset dimensions

But first let's import relevant libraries

#### Imports and Setups

In [1]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")

In [2]:
from pathlib import Path

# Defining path to folder
data_path = Path(r"../01_data/raw")

def load_csv (filename):
    return pd.read_csv(data_path/filename)

field_df = load_csv("Field_Level.csv")
plot_df = load_csv("Plot_Level.csv")
soil_df = load_csv("Soil_PL.csv")

In [3]:

# Create working copies
field_clean = field_df.copy()
plot_clean = plot_df.copy()
soil_clean = soil_df.copy()

# Record baseline dataset dimensions
datasets = {
    "Field_Level": field_clean,
    "Plot_Level": plot_clean,
    "Soil_PL": soil_clean
}

for name, df in datasets.items():
    print(f"\n{'=' * 50}")
    print(name)
    print(f"{'=' * 50}")
    print(f"Rows: {df.shape[0]:,}")
    print(f"Columns: {df.shape[1]:,}")
    print(f"Duplicate rows: {df.duplicated().sum():,}")
    print(f"Total missing cells: {df.isna().sum().sum():,}")


Field_Level
Rows: 399
Columns: 19
Duplicate rows: 0
Total missing cells: 770

Plot_Level
Rows: 1,598
Columns: 27
Duplicate rows: 0
Total missing cells: 1,795

Soil_PL
Rows: 736
Columns: 5
Duplicate rows: 0
Total missing cells: 2,210


In [4]:
# Inspecting the data types and missingness

for name, df in datasets.items():
    print(f"\n--- {name}: data types ---")
    display(df.dtypes.rename("data_type").to_frame())

    print(f"\n--- {name}: missing values ---")
    missing_audit = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_percent": (df.isna().mean() * 100).round(2)
    })

    display(
        missing_audit
        .sort_values("missing_percent", ascending=False)
        .query("missing_count > 0")
    )


--- Field_Level: data types ---


,data_type
block_id,str
season,str
district,str
cell,str
agzone,str
meg_env,str
lat,float64
long,float64
alt,float64
landscape_pos,str



--- Field_Level: missing values ---


,missing_count,missing_percent
trialfield_home_min,281,70.43
slope_pos,157,39.35
soil_depth,157,39.35
rockness,157,39.35
trial_follower_gender,9,2.26
cult_land,9,2.26



--- Plot_Level: data types ---


...

,missing_count,missing_percent
urea_V10_date,1312,82.10
prior_season_crop,460,28.79
silking_date,9,0.56
compost_quality,8,0.50
R3_date,6,0.38



--- Soil_PL: data types ---


,data_type
plot_id,str
pH,float64
tot_N,float64
org_C,float64
sand,float64



--- Soil_PL: missing values ---


,missing_count,missing_percent
tot_N,736,100.00
sand,736,100.00
org_C,736,100.00
pH,2,0.27


### Date Column Preparation

From our audit, we realized that seven plot-level date columns are currently stored as strings 

However, before converting, them, we have to preserve their original values and distinguish whether:

 - A date is valid and can be parsed, or
 - A date is missing in the source

We have to note that a nonmissing value cannot be parsed as a date

**Let's first create parsed date columns**

In [5]:

date_columns = [
    "planting_date",
    "germ_count_date",
    "urea_V6_date",
    "urea_V10_date",
    "silking_date",
    "R3_date",
    "harvest_date"
]

for col in date_columns:
    plot_clean[col + "_parsed"] = pd.to_datetime(
        plot_clean[col],
        errors="coerce"
    )

We opt for `errors="coerce"` so it can convert values that panda cannot parse into NaT (the datetime equivalent of a missing value).

It makes it easy to identify, but it also means we have to distinguish newly created NaT values from missing values initially existing

Now we **audit the conversion results**

In [6]:

date_audit = []

for col in date_columns:
    original_missing = plot_clean[col].isna()
    parsed_missing = plot_clean[col + "_parsed"].isna()

    date_audit.append({
        "column": col,
        "original_missing": original_missing.sum(),
        "parsed_missing": parsed_missing.sum(),
        "new_parse_failures": (
            parsed_missing & ~original_missing
        ).sum(),
        "successfully_parsed": (
            ~parsed_missing
        ).sum()
    })

date_audit = pd.DataFrame(date_audit)

display(date_audit)

,column,original_missing,parsed_missing,new_parse_failures,successfully_parsed
0,planting_date,0,0,0,1598
1,germ_count_date,0,0,0,1598
2,urea_V6_date,0,0,0,1598
3,urea_V10_date,1312,1312,0,286
4,silking_date,9,9,0,1589
5,R3_date,6,6,0,1592
6,harvest_date,0,0,0,1598


- Our new_parse_failures are zero, meaning all nonmissing values were parsed successfully

- If we had parsing failures, we would have needed to inspect those. But we don't